<a href="https://colab.research.google.com/github/kalyanbhumaiahgari/Auto-Analgesia-Safety-System-AASS--Ethical-Solver/blob/main/AASS_Z3_Implementation_Kalyan_%26_Adam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install z3-solver

### 1. Identifying a Deadlock: The Original Contradictory Rules

We define two seemingly reasonable medical rules:

*   **Mercy Rule:** If a patient is in `pain` (`p`), then a `dose` (`d`) should be administered.
*   **Safety Rule:** If a patient has `low_respiration` (`r`), then a `dose` (`d`) should *not* be administered.

We then set up a scenario where both `pain` and `low_respiration` are true simultaneously. Using Z3, we ask if it's possible to satisfy both rules under these conditions. Z3 returns `unsat` (unsatisfiable), which basically means the system would freeze.

This is the deadlock problem.


In [2]:
# The Deadlock
from z3 import *

p = Bool('pain')
r = Bool('low_respiration')
d = Bool('dose')
a = Bool('alert')

solver_deadlock = Solver()

# Original contradictory rules
solver_deadlock.add(Implies(p, d))       # Mercy Rule: if pain, then dose
solver_deadlock.add(Implies(r, Not(d)))  # Safety Rule: if low resp, then no dose

# The conflict case
solver_deadlock.add(p == True)
solver_deadlock.add(r == True)

result = solver_deadlock.check()
print("Deadlock check result:", result)

Deadlock check result: unsat


### 2. Resolving the Deadlock: Revised Logic for Safe Decision Making

logical rules designed to resolve the contradiction and prioritize patient safety.


*   **Dose Administration (`d`):** A `dose` is given *only if* the patient is in `pain` (`p`) AND their `low_respiration` (`r`) is *not* true (i.e., breathing is safe). This ensures the safety rule is never violated.
*   **Alert Activation (`a`):** An `alert` is triggered if the patient is in `pain` (`p`) AND simultaneously experiencing `low_respiration` (`r`). This addresses the previous conflict scenario we had by indicating a need for immediate human intervention rather than automatic dosing.

We apply these new rules to the same conflict case (`pain=True`, `low_respiration=True`) and use Z3 to check for satisfiability and get the outcomes for `dose` and `alert`.

In [3]:
# Revised Logic
solver_revised = Solver()

solver_revised.add(d == And(p, Not(r)))   # Dose only if pain AND safe to breathe
solver_revised.add(a == And(p, r))        # Alert if pain AND respiration is low

# The conflict case — same inputs as before
solver_revised.add(p == True)
solver_revised.add(r == True)

result = solver_revised.check()
print("Revised logic result:", result)

if result == sat:
    m = solver_revised.model()
    print(f"  Dose:  {m[d]}")
    print(f"  Alert: {m[a]}")

Revised logic result: sat
  Dose:  False
  Alert: True


### 3. Comprehensive Scenario Evaluation and Safety Verification

we evaluate here all the possible combinations of the two input variables (`pain` and `low_respiration`). This is a crucial step to confirm that the rules behave as expected across the entire input space and, most importantly, that the critical safety property remains unbroken.

To evaluate the behaviour of the system, we define a helper function called `evaluate_aass`. This function simplifies the process of setting up and running the Z3 solver for each scenario. We then test all four possible combinations of inputs: pain vs. no pain and low respiration vs. normal respiration. For each case, the function determines whether the system decides to administer a `dose` or trigger an `alert`. An important part of this evaluation is checking the safety requirement that medication should never be given when respiration is low.

In [4]:
# Full scenario evaluation (all 4 cases)
from z3 import *

def evaluate_aass(pain_val, resp_val):
    p = Bool('pain')
    r = Bool('low_respiration')
    d = Bool('dose')
    a = Bool('alert')

    s = Solver()
    s.add(d == And(p, Not(r)))
    s.add(a == And(p, r))
    s.add(p == pain_val)
    s.add(r == resp_val)

    assert s.check() == sat, "Logic should always be satisfiable"
    m = s.model()
    return bool(m[d]), bool(m[a])

print(f"{'Pain':<8} {'LowResp':<10} {'Dose':<8} {'Alert':<8} {'Safety OK?'}")
print("-" * 50)

scenarios = [
    (False, False),
    (True,  False),
    (False, True),
    (True,  True),   # The critical conflict case
]

for pain, resp in scenarios:
    dose, alert = evaluate_aass(pain, resp)
    # Safety property: dose must NEVER be True when resp is True
    safety_ok = not (dose and resp)
    print(f"{str(pain):<8} {str(resp):<10} {str(dose):<8} {str(alert):<8} {'✓' if safety_ok else '✗ VIOLATION'}")

print("\nAll safety constraints verified by Z3 SMT solver.")

Pain     LowResp    Dose     Alert    Safety OK?
--------------------------------------------------
False    False      False    False    ✓
True     False      True     False    ✓
False    True       False    False    ✓
True     True       False    True     ✓

All safety constraints verified by Z3 SMT solver.


### 4. Extending the Model: Incorporating Patient Consent

this section shows how the model can be extended to include additional real-world factors, such as patient consent. This introduces a new input variable (`consent`) and a new potential action (`withhold_and_wait`).

The updated rules are:

*   **Dose Administration (`d`):** A `dose` is given only if the patient is in `pain`, their `respiration` is safe (`Not(r)`), *and* `consent` has been given.
*   **Alert Activation (`a`):** An `alert` is still triggered if there's `pain` and `low_respiration` (as this remains a critical situation irrespective of consent).
*   **Withhold and Wait (`w`):** This new action is triggered if the patient is in `pain`, `respiration` is safe, *but* `consent` has *not* been given. This represents a scenario where the patient is stable enough to wait for consent or further assessment.

We then evaluate several key scenarios, including the new consent-related cases, to observe how the extended logic behaves and to confirm the new `withhold_and_wait` action is correctly triggered.

In [5]:
# Extended model with 3 inputs (added patient consent)
from z3 import *

p = Bool('pain')           # patient is in pain
r = Bool('low_respiration')  # dangerous respiration
c = Bool('consent')        # patient (or proxy) has given consent

d = Bool('dose')
a = Bool('alert')
w = Bool('withhold_and_wait')  # new action: stable but withhold

s = Solver()

# Revised rules with consent
# Dose: pain present, breathing safe, AND consent given
s.add(d == And(p, Not(r), c))

# Alert: pain AND dangerous respiration (regardless of consent)
s.add(a == And(p, r))

# Withhold-and-wait: pain but no consent, and breathing is safe
s.add(w == And(p, Not(r), Not(c)))

scenarios = [
    (True, False, True,  "Normal case — dose expected"),
    (True, True,  True,  "Conflict case — alert, no dose"),
    (True, False, False, "No consent — withhold and wait"),
    (False, False, True, "No pain — do nothing"),
]

print(f"{'Pain':<6} {'LowR':<6} {'Cons':<6} {'Dose':<6} {'Alert':<7} {'Wait':<6} Scenario")
print("-" * 70)

for pain_v, resp_v, cons_v, label in scenarios:
    s2 = Solver()
    s2.add(d == And(p, Not(r), c))
    s2.add(a == And(p, r))
    s2.add(w == And(p, Not(r), Not(c)))
    s2.add(p == pain_v, r == resp_v, c == cons_v)

    assert s2.check() == sat
    m = s2.model()
    print(f"{str(pain_v):<6} {str(resp_v):<6} {str(cons_v):<6} "
          f"{str(bool(m[d])):<6} {str(bool(m[a])):<7} {str(bool(m[w])):<6} {label}")

Pain   LowR   Cons   Dose   Alert   Wait   Scenario
----------------------------------------------------------------------
True   False  True   True   False   False  Normal case — dose expected
True   True   True   False  True    False  Conflict case — alert, no dose
True   False  False  False  False   True   No consent — withhold and wait
False  False  True   False  False   False  No pain — do nothing


In [7]:
"""
can dose and alert ever both be True
at the same time? That would be a contradiction — you can't be dosing a
patient AND alerting a nurse that dosing is dangerous simultaneously.
We added this check ourselves to verify they're mutually exclusive.

"""

from z3 import *
p, r, d, a = Bools('pain low_respiration dose alert')
s = Solver()
s.add(d == And(p, Not(r)))
s.add(a == And(p, r))
s.add(d == True)
s.add(a == True)
print("Can dose and alert both fire simultaneously?", s.check())


Can dose and alert both fire simultaneously? unsat


In [9]:
# Extending the model — adding nurse response and escalation to test the
# liveness property described in Section IV.
#
# The idea here is , what if an alert fires but no nurse shows up?
# We need the system to escalate rather than silently do nothing.
# n = nurse has acknowledged the alert
# e = escalate to senior staff / emergency if alert active and nurse is absent

import itertools
from z3 import *

# Picking up all variables, as previously we redeclared p,r,d,a without c,w
p = Bool('pain')
r = Bool('low_respiration')
c = Bool('consent')
n = Bool('nurse_ok')
d = Bool('dose')
a = Bool('alert')
w = Bool('withhold_and_wait')
e = Bool('escalate')

s = Solver()
s.add(d == And(p, Not(r), c))
s.add(a == And(p, r))
s.add(w == And(p, Not(r), Not(c)))
s.add(e == And(a, Not(n)))

# Liveness proof: can alert fire, nurse be absent, and escalation NOT fire?
# If Z3 returns unsat, the system is provably live — it can never stall.
s.push()
s.add(And(a == True, n == False, e == False))
print("Liveness Proof (Escalation when nurse absent):", s.check())
s.pop()

# Full evaluation across all 8 input combinations x 2 nurse states

print(f"\n{'Pain':<7}{'LowR':<7}{'Cons':<7}{'Nurse':<7}{'Dose':<7}{'Alert':<7}{'Wait':<7}{'Esc':<7}Safety")
print("-" * 65)

for pain_val, resp_val, cons_val in itertools.product([True, False], repeat=3):
    for nurse_val in [True, False]:
        s2 = Solver()
        s2.add(d == And(p, Not(r), c))
        s2.add(a == And(p, r))
        s2.add(w == And(p, Not(r), Not(c)))
        s2.add(e == And(a, Not(n)))
        s2.add(p == pain_val, r == resp_val, c == cons_val, n == nurse_val)
        assert s2.check() == sat
        m = s2.model()
        safety_ok = not (bool(m[d]) and resp_val)
        print(f"{str(pain_val):<7}{str(resp_val):<7}{str(cons_val):<7}"
              f"{str(nurse_val):<7}{str(bool(m[d])):<7}{str(bool(m[a])):<7}"
              f"{str(bool(m[w])):<7}{str(bool(m[e])):<7}{'✓' if safety_ok else '✗'}")



Liveness Proof (Escalation when nurse absent): unsat

Pain   LowR   Cons   Nurse  Dose   Alert  Wait   Esc    Safety
-----------------------------------------------------------------
True   True   True   True   False  True   False  False  ✓
True   True   True   False  False  True   False  True   ✓
True   True   False  True   False  True   False  False  ✓
True   True   False  False  False  True   False  True   ✓
True   False  True   True   True   False  False  False  ✓
True   False  True   False  True   False  False  False  ✓
True   False  False  True   False  False  True   False  ✓
True   False  False  False  False  False  True   False  ✓
False  True   True   True   False  False  False  False  ✓
False  True   True   False  False  False  False  False  ✓
False  True   False  True   False  False  False  False  ✓
False  True   False  False  False  False  False  False  ✓
False  False  True   True   False  False  False  False  ✓
False  False  True   False  False  False  False  False  ✓
False